In [ ]:
import shutil
import os

# 1. Mount Drive

# 2. Unzip
zip_path = './data/fraud_detection_backup.zip'
extract_path = './data/fraud_detection'

shutil.unpack_archive(zip_path, extract_path)
print("Unzipped to:", extract_path)

# 3. Reload everything
import joblib
import pandas as pd

lean_model   = joblib.load(f'{extract_path}/lean_model.pkl')
X_train_lean = pd.read_csv(f'{extract_path}/X_train_lean.csv').drop(columns='Unnamed: 0')
X_test_lean  = pd.read_csv(f'{extract_path}/X_test_lean.csv').drop(columns='Unnamed: 0')
y_train      = pd.read_csv(f'{extract_path}/y_train.csv').drop(columns='Unnamed: 0').squeeze()
y_test       = pd.read_csv(f'{extract_path}/y_test.csv').drop(columns='Unnamed: 0').squeeze()

print("\nLoaded successfully:")
print(f"  X_train_lean : {X_train_lean.shape}")
print(f"  X_test_lean  : {X_test_lean.shape}")
print(f"  y_train      : {y_train.shape}")
print(f"  y_test       : {y_test.shape}")
print(f"  lean_model   : {type(lean_model)}")

Unzipped to: /content/fraud_detection

Loaded successfully:
  X_train_lean : (445173, 311)
  X_test_lean  : (145367, 311)
  y_train      : (445173,)
  y_test       : (145367,)
  lean_model   : <class 'xgboost.sklearn.XGBClassifier'>


In [ ]:
import numpy as np

In [ ]:
pip install xgboost

In [ ]:
d_cols = [c for c in X_train_lean.columns if c.startswith('D') and c[1:].isdigit()]

for col in d_cols:
    for df_ in [X_train_lean, X_test_lean]:
        df_[f'{col}_normalized'] = (df_[col] - df_['TransactionDT'] / 86400).astype('float32')

In [ ]:
# Step 1: Create intermediate combined column
X_train_lean['card1_addr1'] = (X_train_lean['card1_te'].astype(str) + '_' +
                                X_train_lean['addr1_te'].astype(str))
X_test_lean['card1_addr1']  = (X_test_lean['card1_te'].astype(str) + '_' +
                                X_test_lean['addr1_te'].astype(str))

# Step 2: Create day column
X_train_lean['day'] = X_train_lean['TransactionDT'] / (24*60*60)
X_test_lean['day']  = X_test_lean['TransactionDT'] / (24*60*60)

# Step 3: Create magic UID
X_train_lean['uid_magic'] = (X_train_lean['card1_addr1'].astype(str) + '_' +
                              np.floor(X_train_lean['day'] - X_train_lean['D1']).astype(str))
X_test_lean['uid_magic']  = (X_test_lean['card1_addr1'].astype(str) + '_' +
                              np.floor(X_test_lean['day'] - X_test_lean['D1']).astype(str))

# # Step 4: Aggregate features per magic UID (fit on train only)
# for agg_col in ['TransactionAmt', 'D4', 'D10', 'D15']:
#     uid_mean = X_train_lean.groupby('uid_magic')[agg_col].mean()
#     uid_std  = X_train_lean.groupby('uid_magic')[agg_col].std()
#     print(uid_mean, uid_std)
#     X_train_lean[f'{agg_col}_uid_mean'] = X_train_lean['uid_magic'].map(uid_mean).astype('float32')
#     X_train_lean[f'{agg_col}_uid_std']  = X_train_lean['uid_magic'].map(uid_std).astype('float32')
#     X_test_lean[f'{agg_col}_uid_mean']  = X_test_lean['uid_magic'].map(uid_mean).astype('float32')
#     X_test_lean[f'{agg_col}_uid_std']   = X_test_lean['uid_magic'].map(uid_std).astype('float32')

# Step 5: cents and outsider15
X_train_lean['cents'] = (X_train_lean['TransactionAmt'] -
                          np.floor(X_train_lean['TransactionAmt'])).astype('float32')
X_test_lean['cents']  = (X_test_lean['TransactionAmt'] -
                          np.floor(X_test_lean['TransactionAmt'])).astype('float32')

X_train_lean['outsider15'] = (np.abs(X_train_lean['D1'] - X_train_lean['D15']) > 3).astype('int8')
X_test_lean['outsider15']  = (np.abs(X_test_lean['D1']  - X_test_lean['D15'])  > 3).astype('int8')

# Step 6: Drop intermediate columns not needed as features
X_train_lean.drop(columns=['card1_addr1', 'day'], inplace=True)
X_test_lean.drop(columns=['card1_addr1', 'day'], inplace=True)

print(f"New features added. Shape: {X_train_lean.shape}")

New features added. Shape: (445173, 327)


In [ ]:
import datetime
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import numpy as np

# Step 1: Combine data
X_all = pd.concat([X_train_lean, X_test_lean], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

# Step 2: Create month groups from TransactionDT
START_DATE = datetime.datetime.strptime('2017-11-30', '%Y-%m-%d')
X_all['DT_M'] = X_all['TransactionDT'].apply(
    lambda x: (START_DATE + datetime.timedelta(seconds=x))
)
X_all['DT_M'] = (X_all['DT_M'].dt.year - 2017) * 12 + X_all['DT_M'].dt.month

print("Month distribution:")
print(X_all['DT_M'].value_counts().sort_index())

Month distribution:
DT_M
12    137321
13     92585
14     86021
15    101632
16     83655
17     89326
Name: count, dtype: int64


In [ ]:
# Drop uid_magic from X_all before the GroupKFold loop
cols_to_drop = ['uid_magic', 'DT_M', 'TransactionDT']
feature_cols = [c for c in X_all.columns if c not in cols_to_drop]

print(f"Training features: {len(feature_cols)}")

# Check for any remaining object columns
obj_cols = X_all[feature_cols].select_dtypes('object').columns.tolist()
if obj_cols:
    print(f"WARNING - object columns still present: {obj_cols}")
    feature_cols = [c for c in feature_cols if c not in obj_cols]
    print(f"Dropped {len(obj_cols)} object columns. Remaining: {len(feature_cols)}")
else:
    print("No object columns — ready to train.")

Training features: 325
No object columns — ready to train.


In [ ]:
# Find all object columns in X_all
obj_cols = X_all.select_dtypes('object').columns.tolist()
print(f"Object columns found: {obj_cols}")

# Drop them permanently from X_all
X_all.drop(columns=obj_cols, inplace=True)
print(f"Dropped. Remaining shape: {X_all.shape}")

# Verify no object columns remain
remaining_obj = X_all.select_dtypes('object').columns.tolist()
print(f"Remaining object columns: {remaining_obj}")

# Update feature_cols
feature_cols = [c for c in X_all.columns if c not in ['DT_M', 'TransactionDT']]
print(f"Feature cols for training: {len(feature_cols)}")

Object columns found: ['uid_magic']
Dropped. Remaining shape: (590540, 327)
Remaining object columns: []
Feature cols for training: 325


In [ ]:
from collections import defaultdict
import xgboost as xgb

In [ ]:
# Step 3: Run GroupKFold
oof_probabilities_Group = np.zeros(len(X_all))
best_iterations = []
fold_importances = defaultdict(list)

# Drop DT_M before training — it's only used for grouping
feature_cols = [c for c in X_all.columns if c not in ['DT_M', 'TransactionDT']]

skf = GroupKFold(n_splits=6)
groups = X_all['DT_M']

print("--- Starting GroupKFold Evaluation ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all, groups=groups)):
    X_tr = X_all[feature_cols].iloc[train_idx]
    X_va = X_all[feature_cols].iloc[val_idx]
    y_tr = y_all.iloc[train_idx]
    y_va = y_all.iloc[val_idx]

    month = X_all['DT_M'].iloc[val_idx].iloc[0]
    print(f"Fold {fold+1} — withholding month {month} | train: {len(train_idx)} rows, val: {len(val_idx)} rows")

    fold_model = xgb.XGBClassifier(
        max_depth=12,
        learning_rate=0.0257,
        n_estimators=2000,
        min_child_weight=10,
        subsample=0.9844,
        colsample_bytree=0.6017,
        gamma=0.2863,
        reg_alpha=0.6264,
        reg_lambda=0.1137,
        eval_metric='aucpr',
        device='cuda',
        early_stopping_rounds=150
    )

    fold_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    best_iterations.append(fold_model.best_iteration)
    oof_probabilities_Group[val_idx] = fold_model.predict_proba(X_va)[:, 1]

    # Collect importance
    booster = fold_model.get_booster()
    importance = booster.get_score(importance_type='gain')
    for feat, score in importance.items():
        fold_importances[feat].append(score)

    fold_auc = roc_auc_score(y_va, oof_probabilities_Group[val_idx])
    print(f"  Best iteration: {fold_model.best_iteration} | Fold AUC: {fold_auc:.4f}")

print(f"\nOverall OOF AUC: {roc_auc_score(y_all, oof_probabilities_Group):.4f}")
avg_trees = int(np.mean(best_iterations))
print(f"Average optimal trees: {avg_trees}")

--- Starting GroupKFold Evaluation ---
Fold 1 — withholding month 12 | train: 453219 rows, val: 137321 rows


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [18:31:16] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


  Best iteration: 1999 | Fold AUC: 0.9199
Fold 2 — withholding month 15 | train: 488908 rows, val: 101632 rows
  Best iteration: 1785 | Fold AUC: 0.9469
Fold 3 — withholding month 13 | train: 497955 rows, val: 92585 rows
  Best iteration: 1985 | Fold AUC: 0.9524
Fold 4 — withholding month 17 | train: 501214 rows, val: 89326 rows
  Best iteration: 1063 | Fold AUC: 0.9320
Fold 5 — withholding month 14 | train: 504519 rows, val: 86021 rows
  Best iteration: 1986 | Fold AUC: 0.9531
Fold 6 — withholding month 16 | train: 506885 rows, val: 83655 rows
  Best iteration: 1951 | Fold AUC: 0.9585

Overall OOF AUC: 0.9436
Average optimal trees: 1794


In [ ]:
# Find all object columns in X_all
obj_cols = X_all.select_dtypes('object').columns.tolist()
print(f"Object columns found: {obj_cols}")

# Drop them permanently from X_all
X_all.drop(columns=obj_cols, inplace=True)
print(f"Dropped. Remaining shape: {X_all.shape}")

# Verify no object columns remain
remaining_obj = X_all.select_dtypes('object').columns.tolist()
print(f"Remaining object columns: {remaining_obj}")

# Update feature_cols
feature_cols = [c for c in X_all.columns if c not in ['DT_M', 'TransactionDT']]
print(f"Feature cols for training: {len(feature_cols)}")

Object columns found: []
Dropped. Remaining shape: (590540, 327)
Remaining object columns: []
Feature cols for training: 325


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
import xgboost as xgb


# 2. Initialize an empty array to store the model's validation probabilities
# This will hold the "unbiased test predictions" for the whole dataset
oof_probabilities_Strat = np.zeros(len(X_all))

# Configure Stratified 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_iterations = []

print("--- Starting Stratified K-Fold Evaluation ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    # Slice into this fold's training and evaluation sets
    X_tr, X_va = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_va = y_all.iloc[train_idx], y_all.iloc[val_idx]

    # Initialize your tuned parameters
    fold_model = xgb.XGBClassifier(
        max_depth=12,
        learning_rate=0.0257,
        n_estimators=2000,
        min_child_weight=10,
        subsample=0.9844,
        colsample_bytree=0.6017,
        gamma=0.2863,
        reg_alpha=0.6264,
        reg_lambda=0.1137,
        eval_metric='aucpr',
        device='cuda',
        early_stopping_rounds=150
    )

    # Train the fold
    fold_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    # Track the best tree iteration for this fold
    best_iterations.append(fold_model.best_iteration)

    # TEST THE MODEL: Predict on the validation fold (data the model hasn't seen yet)
    # and save those probabilities into our global array
    oof_probabilities_Strat[val_idx] = fold_model.predict_proba(X_va)[:, 1]

    print(f"Fold {fold + 1} complete. Best iteration: {fold_model.best_iteration}")

print("\n--- Evaluation Phase ---")

--- Starting Stratified K-Fold Evaluation ---
Fold 1 complete. Best iteration: 1993
Fold 2 complete. Best iteration: 1999
Fold 3 complete. Best iteration: 1998
Fold 4 complete. Best iteration: 1997
Fold 5 complete. Best iteration: 1999

--- Evaluation Phase ---


In [ ]:
# Step 4: Threshold scan
from sklearn.metrics import f1_score, precision_score, recall_score

for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.5, 0.55, 0.6]:
    preds = (oof_probabilities_Group >= t).astype(int)
    p = precision_score(y_all, preds)
    r = recall_score(y_all, preds)
    f = f1_score(y_all, preds)
    print(f"Threshold {t:.2f} → P:{p:.3f} R:{r:.3f} F1:{f:.3f}")

Threshold 0.20 → P:0.690 R:0.607 F1:0.646
Threshold 0.25 → P:0.733 R:0.579 F1:0.647
Threshold 0.30 → P:0.768 R:0.555 F1:0.644
Threshold 0.35 → P:0.796 R:0.534 F1:0.640
Threshold 0.40 → P:0.819 R:0.515 F1:0.632
Threshold 0.45 → P:0.838 R:0.496 F1:0.623
Threshold 0.50 → P:0.856 R:0.476 F1:0.612
Threshold 0.55 → P:0.871 R:0.459 F1:0.601
Threshold 0.60 → P:0.884 R:0.441 F1:0.588


In [ ]:
# Test different decision thresholds to evaluate precision-recall behavior
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

for t in thresholds:
    preds = (oof_probabilities_Strat >= t).astype(int)

    # Calculate metrics manually for evaluation
    tp = np.sum((preds == 1) & (y_all == 1))
    fp = np.sum((preds == 1) & (y_all == 0))
    fn = np.sum((preds == 0) & (y_all == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Threshold {t:.2f} → Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

Threshold 0.20 → Precision: 0.847, Recall: 0.764, F1: 0.803
Threshold 0.25 → Precision: 0.884, Recall: 0.743, F1: 0.807
Threshold 0.30 → Precision: 0.907, Recall: 0.721, F1: 0.804
Threshold 0.35 → Precision: 0.928, Recall: 0.705, F1: 0.801
Threshold 0.40 → Precision: 0.943, Recall: 0.687, F1: 0.795
Threshold 0.45 → Precision: 0.953, Recall: 0.669, F1: 0.786
Threshold 0.50 → Precision: 0.961, Recall: 0.652, F1: 0.777
Threshold 0.55 → Precision: 0.968, Recall: 0.632, F1: 0.765


In [ ]:
# Then ensemble
for w_gkf in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    w_skf = 1 - w_gkf
    ensemble_probs = (w_gkf * oof_probabilities_Group) + (w_skf * oof_probabilities_Strat)

    for t in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]:
        preds = (ensemble_probs >= t).astype(int)
        p = precision_score(y_all, preds)
        r = recall_score(y_all, preds)
        f = f1_score(y_all, preds)
        auc = roc_auc_score(y_all, ensemble_probs)
        print(f"GKF:{w_gkf} SKF:{w_skf} t:{t} → P:{p:.3f} R:{r:.3f} F1:{f:.3f} AUC:{auc:.4f}")

GKF:0.3 SKF:0.7 t:0.2 → P:0.816 R:0.751 F1:0.782 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.25 → P:0.860 R:0.726 F1:0.787 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.3 → P:0.892 R:0.703 F1:0.786 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.35 → P:0.915 R:0.681 F1:0.781 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.4 → P:0.933 R:0.659 F1:0.772 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.45 → P:0.946 R:0.636 F1:0.761 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.5 → P:0.956 R:0.613 F1:0.747 AUC:0.9721
GKF:0.3 SKF:0.7 t:0.55 → P:0.963 R:0.590 F1:0.732 AUC:0.9721
GKF:0.4 SKF:0.6 t:0.2 → P:0.803 R:0.744 F1:0.772 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.25 → P:0.846 R:0.718 F1:0.777 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.3 → P:0.878 R:0.693 F1:0.774 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.35 → P:0.903 R:0.668 F1:0.768 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.4 → P:0.924 R:0.645 F1:0.759 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.45 → P:0.938 R:0.619 F1:0.746 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.5 → P:0.950 R:0.593 F1:0.730 AUC:0.9713
GKF:0.4 SKF:0.6 t:0.55 → P:0.959 R:0.562 F1:0.709 AUC:0.9713
GKF:0.5 SKF:0.5 t:0.2 → P:0.791 

In [ ]:
import joblib
import json
from datetime import datetime

# --- Retrain on full data using avg best iteration ---
avg_trees_strat = int(np.mean(best_iterations))
print(f"Retraining Stratified model with {avg_trees_strat} trees on full dataset...")

model_strat = xgb.XGBClassifier(
    max_depth=12,
    learning_rate=0.0257,
    n_estimators=avg_trees_strat,   # fixed — no early stopping needed
    min_child_weight=10,
    subsample=0.9844,
    colsample_bytree=0.6017,
    gamma=0.2863,
    reg_alpha=0.6264,
    reg_lambda=0.1137,
    eval_metric='aucpr',
    device='cuda',
)

model_strat.fit(X_all, y_all, verbose=False)

# Save model + metadata
model_strat.save_model("model_strat.ubj")   # native XGBoost binary (fastest)
joblib.dump(model_strat, "model_strat.pkl") # sklearn-compatible pickle

metadata_strat = {
    "saved_at": datetime.now().isoformat(),
    "n_estimators": avg_trees_strat,
    "oof_auc": roc_auc_score(y_all, oof_probabilities_Strat),
    "threshold": 0.25,
    "cv_strategy": "StratifiedKFold",
    "n_splits": 5,
    "features": list(X_all.columns),
}
with open("model_strat_metadata.json", "w") as f:
    json.dump(metadata_strat, f, indent=2)

print(f"Saved: model_strat.ubj | OOF AUC: {metadata_strat['oof_auc']:.4f}")

Retraining Stratified model with 1995 trees on full dataset...
Saved: model_strat.ubj | OOF AUC: 0.9733


In [ ]:
avg_trees_group = 1934
print(f"Retraining Group model with {avg_trees_group} trees on full dataset...")

model_group = xgb.XGBClassifier(
    max_depth=12,
    learning_rate=0.0257,
    n_estimators=avg_trees_group,
    min_child_weight=10,
    subsample=0.9844,
    colsample_bytree=0.6017,
    gamma=0.2863,
    reg_alpha=0.6264,
    reg_lambda=0.1137,
    eval_metric='aucpr',
    device='cuda',
)

model_group.fit(X_all[feature_cols], y_all, verbose=False)

model_group.save_model("model_group.ubj")
joblib.dump(model_group, "model_group.pkl")

metadata_group = {
    "saved_at": datetime.now().isoformat(),
    "n_estimators": avg_trees_group,
    "oof_auc": roc_auc_score(y_all, oof_probabilities_Group),
    "threshold": 0.25,
    "cv_strategy": "GroupKFold",
    "n_splits": 6,
    "group_col": "DT_M",
    "features": feature_cols,
}
with open("model_group_metadata.json", "w") as f:
    json.dump(metadata_group, f, indent=2)

print(f"Saved: model_group.ubj | OOF AUC: {metadata_group['oof_auc']:.4f}")

Retraining Group model with 1934 trees on full dataset...
Saved: model_group.ubj | OOF AUC: 0.9440


In [ ]:
import lightgbm as lgb
from sklearn.metrics import precision_score, recall_score

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.07,
    num_leaves=63,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42,
    verbose=100
)
lgb_model.fit(
    X_train_lean, y_train,
    eval_set=[(X_test_lean, y_test)],
)

y_prob_lgb = lgb_model.predict_proba(X_test_lean)[:, 1]
for t in [0.20, 0.25, 0.30, 0.35, 0.40]:
    y_pred = (y_prob_lgb >= t).astype(int)
    p = precision_score(y_test, y_pred)
    r = recall_score(y_test, y_pred)
    print(f"Threshold {t:.2f} → Precision: {p:.3f}, Recall: {r:.3f}")

ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: uid_magic: object

In [ ]:
y_prob_xgb = lean_model.predict_proba(X_test_lean)[:, 1]
y_prob_lgb = lgb_model.predict_proba(X_test_lean)[:, 1]

for xgb_w in [0.7, 0.8, 0.9]:
    lgb_w = 1 - xgb_w
    y_prob_final = (xgb_w * y_prob_xgb) + (lgb_w * y_prob_lgb)
    for t in [0.35, 0.40, 0.45]:
        y_pred = (y_prob_final >= t).astype(int)
        p = precision_score(y_test, y_pred)
        r = recall_score(y_test, y_pred)
        f = 2 * p * r / (p + r)
        print(f"XGB:{xgb_w} LGB:{lgb_w} t:{t} → P:{p:.3f} R:{r:.3f} F1:{f:.3f}")

XGB:0.7 LGB:0.30000000000000004 t:0.35 → P:0.675 R:0.552 F1:0.607
XGB:0.7 LGB:0.30000000000000004 t:0.4 → P:0.706 R:0.533 F1:0.608
XGB:0.7 LGB:0.30000000000000004 t:0.45 → P:0.733 R:0.518 F1:0.607
XGB:0.8 LGB:0.19999999999999996 t:0.35 → P:0.681 R:0.547 F1:0.607
XGB:0.8 LGB:0.19999999999999996 t:0.4 → P:0.712 R:0.531 F1:0.608
XGB:0.8 LGB:0.19999999999999996 t:0.45 → P:0.738 R:0.517 F1:0.608
XGB:0.9 LGB:0.09999999999999998 t:0.35 → P:0.687 R:0.546 F1:0.609
XGB:0.9 LGB:0.09999999999999998 t:0.4 → P:0.715 R:0.531 F1:0.609
XGB:0.9 LGB:0.09999999999999998 t:0.45 → P:0.739 R:0.518 F1:0.609


Threshold 0.20 → Precision: 0.232, Recall: 0.785
Threshold 0.25 → Precision: 0.272, Recall: 0.753
Threshold 0.30 → Precision: 0.311, Recall: 0.728
Threshold 0.35 → Precision: 0.350, Recall: 0.701
Threshold 0.40 → Precision: 0.387, Recall: 0.673

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_score, recall_score


def time_series_split(df: pd.DataFrame, target_col: str, split_ratio: float = 0.8):
    """
    Splits a DataFrame chronologically into Train and Test sets.
    Assumes the DataFrame index is sorted by time.
    """
    split_idx = int(len(df) * split_ratio)

    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    return X_train, X_test, y_train, y_test


def train_xgboost_model(X_train: pd.DataFrame, y_train: pd.Series,
                        X_test: pd.DataFrame, y_test: pd.Series,
                        model_type: str = 'regression',
                        params: dict = None) -> xgb.XGBModel:
    """
    Trains an XGBoost model with early stopping to prevent overfitting.
    Dynamically adjusts scale_pos_weight and eval_metric for classification.
    """
    # Default high-performing hyperparameters
    default_params = {
      'n_estimators': 2000,
      'learning_rate': 0.03,       # slower learning, better generalization
      'max_depth': 8,              # deeper trees
      'subsample': 0.8,
      'colsample_bytree': 0.8,
      'min_child_weight': 5,       # prevents overfitting on rare fraud cases
      'gamma': 0.1,                # minimum loss reduction for split
      'reg_alpha': 0.1,            # L1 regularization
      'reg_lambda': 1.0,           # L2 regularization
      'random_state': 42,
      'early_stopping_rounds': 100,
      'tree_method': 'hist',
      'device': 'cuda',
      'eval_metric': 'aucpr'
    }

    # Classification-specific adjustments for class imbalance
    if model_type == 'classification':
        # Calculate scale_pos_weight dynamically: count(negative) / count(positive)
        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()

        if pos_count > 0:
            calculated_weight = neg_count / pos_count
            default_params['scale_pos_weight'] = calculated_weight

        # Use Area Under Precision-Recall Curve for class-imbalanced tracking
        default_params['eval_metric'] = 'aucpr'

    # Override defaults if custom parameters are passed
    if params:
        default_params.update(params)

    # Initialize the correct model archetype
    if model_type == 'regression':
        model = xgb.XGBRegressor(**default_params)
    elif model_type == 'classification':
        model = xgb.XGBClassifier(**default_params)
    else:
        raise ValueError("model_type must be either 'regression' or 'classification'")

    # Train the model using the test set as validation for early stopping
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=100  # Prints progress every 100 trees
    )

    return model


def evaluate_classification(model, X_test, y_test):
    """
    Evaluates a classification model and returns key performance metrics.
    """
    # Hard predictions (0 or 1)
    predictions = model.predict(X_test)

    # Probability predictions (needed for ROC-AUC)
    probabilities = model.predict_proba(X_test)[:, 1]
    for t in [0.20, 0.25, 0.30, 0.35,0.40, 0.45, 0.50, 0.55]:
      y_pred = (probabilities >= t).astype(int)
      p = precision_score(y_test, y_pred)
      r = recall_score(y_test, y_pred)
      f = 2 * p * r / (p + r)
      print(f"Threshold {t:.2f} → Precision: {p:.3f}, Recall: {r:.3f}, F1: {f:.3f}")

    accuracy = accuracy_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    print("--- Classification Report ---")
    print(classification_report(y_test, predictions))

    metrics = {
        "Accuracy": accuracy,
        "ROC_AUC": roc_auc
    }
    return metrics


def save_pipeline(model: xgb.XGBModel, filepath: str):
    """Saves the trained model weights to disk."""
    joblib.dump(model, filepath)
    print(f"Model successfully saved to {filepath}")


def load_pipeline(filepath: str) -> xgb.XGBModel:
    """Loads a trained model weights from disk."""
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No model found at {filepath}")
    return joblib.load(filepath)

In [ ]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00


In [ ]:
from catboost import CatBoostClassifier

cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=8,
    auto_class_weights='Balanced',  # let CatBoost compute it
    eval_metric='PRAUC',
    loss_function='Logloss',
    random_seed=42,
    verbose=100
)

cat_model.fit(X_train_lean, y_train, eval_set=(X_test_lean, y_test))

# 4. Evaluate performance
metrics = evaluate_classification(cat_model, X_test_lean, y_test)
print("\nEvaluation Metrics:", metrics)

0:	learn: 0.8810325	test: 0.8698247	best: 0.8698247 (0)	total: 584ms	remaining: 19m 27s
100:	learn: 0.9405396	test: 0.9197278	best: 0.9197308 (99)	total: 1m 8s	remaining: 21m 25s
200:	learn: 0.9517898	test: 0.9248462	best: 0.9248462 (200)	total: 2m 14s	remaining: 20m 4s
300:	learn: 0.9624445	test: 0.9302769	best: 0.9302769 (300)	total: 3m 21s	remaining: 18m 56s
400:	learn: 0.9713917	test: 0.9333200	best: 0.9333209 (399)	total: 4m 31s	remaining: 18m 3s
500:	learn: 0.9774020	test: 0.9350641	best: 0.9350728 (499)	total: 5m 41s	remaining: 17m 2s
600:	learn: 0.9818178	test: 0.9358218	best: 0.9358368 (594)	total: 6m 53s	remaining: 16m 1s
700:	learn: 0.9852958	test: 0.9363251	best: 0.9363817 (697)	total: 8m 3s	remaining: 14m 56s
800:	learn: 0.9879375	test: 0.9362335	best: 0.9364021 (719)	total: 9m 13s	remaining: 13m 48s
900:	learn: 0.9900495	test: 0.9364868	best: 0.9365118 (888)	total: 10m 25s	remaining: 12m 43s
1000:	learn: 0.9916848	test: 0.9364871	best: 0.9366816 (956)	total: 11m 36s	remai

In [ ]:
d_cols = [c for c in X_train_lean.columns if c.startswith('D') and c[1:].isdigit()]

for col in d_cols:
    for df_ in [X_train_lean, X_test_lean]:
        df_[f'{col}_normalized'] = (df_[col] - df_['TransactionDT'] / 86400).astype('float32')

In [ ]:
params = {
    'max_depth':16,
    'min_child_weight': 3,
    'gamma': 0.2,
    'subsample': 0.7,
    'colsample_bytree': 0.7,
}

# lean_model = train_xgboost_model(
#     X_train_lean, y_train,
#     X_test_lean, y_test,
#     model_type='classification',
#     params=params
# )

# 4. Evaluate performance
metrics = evaluate_classification(lean_model, X_test_lean, y_test)
print("\nEvaluation Metrics:", metrics)

Threshold 0.20 → Precision: 0.622, Recall: 0.595, F1: 0.608
Threshold 0.25 → Precision: 0.665, Recall: 0.570, F1: 0.613
Threshold 0.30 → Precision: 0.694, Recall: 0.552, F1: 0.615
Threshold 0.35 → Precision: 0.722, Recall: 0.536, F1: 0.615
Threshold 0.40 → Precision: 0.744, Recall: 0.522, F1: 0.614
Threshold 0.45 → Precision: 0.765, Recall: 0.510, F1: 0.612
Threshold 0.50 → Precision: 0.782, Recall: 0.498, F1: 0.608
Threshold 0.55 → Precision: 0.800, Recall: 0.487, F1: 0.606
--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    140334
           1       0.78      0.50      0.61      5033

    accuracy                           0.98    145367
   macro avg       0.88      0.75      0.80    145367
weighted avg       0.98      0.98      0.98    145367


Evaluation Metrics: {'Accuracy': 0.9778078931256751, 'ROC_AUC': np.float64(0.9336617638364397)}


In [ ]:
# After retraining XGBoost, run this:
booster = lean_model.get_booster()
importance_dict = booster.get_score(importance_type='gain')

low_importance = [f for f, score in importance_dict.items() if score < 15.0]
noise_flags = [c for c in X_train_lean.columns
               if c.endswith('_missing') and X_train_lean[c].mean() > 0.85]

drop_cols = list(set(low_importance + noise_flags))
X_train_lean.drop(columns=[c for c in drop_cols if c in X_train_lean.columns], inplace=True)
X_test_lean.drop(columns=[c for c in drop_cols if c in X_test_lean.columns], inplace=True)
print(f"Dropped {len(drop_cols)} noisy features. Remaining: {X_train_lean.shape[1]}")

Dropped 6 noisy features. Remaining: 230


In [ ]:
# pip install optuna
# import optuna
# from sklearn.metrics import f1_score
# optuna.logging.set_verbosity(optuna.logging.WARNING)

# def objective(trial):
#     params = {
#         'max_depth':        trial.suggest_int('max_depth', 6, 12),
#         'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
#         'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
#         'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
#         'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 2.0),
#     }
#     model = train_xgboost_model(
#         X_train_lean, y_train,
#         X_test_lean, y_test,
#         model_type='classification',
#         params=params
#     )
#     y_prob = model.predict_proba(X_test_lean)[:, 1]
#     y_pred = (y_prob >= 0.45).astype(int)
#     return f1_score(y_test, y_pred)

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=30, show_progress_bar=True)

# print("Best F1:", study.best_value)
# print("Best params:", study.best_params)

  0%|          | 0/30 [00:00<?, ?it/s]

[0]	validation_0-aucpr:0.36887
[100]	validation_0-aucpr:0.56483
[200]	validation_0-aucpr:0.58798
[300]	validation_0-aucpr:0.59975
[400]	validation_0-aucpr:0.60709
[500]	validation_0-aucpr:0.61212
[600]	validation_0-aucpr:0.61588
[700]	validation_0-aucpr:0.61875
[800]	validation_0-aucpr:0.62045
[900]	validation_0-aucpr:0.62271
[1000]	validation_0-aucpr:0.62512
[1100]	validation_0-aucpr:0.62695
[1200]	validation_0-aucpr:0.62790
[1300]	validation_0-aucpr:0.62929
[1400]	validation_0-aucpr:0.63099
[1500]	validation_0-aucpr:0.63219
[1600]	validation_0-aucpr:0.63343
[1700]	validation_0-aucpr:0.63440
[1800]	validation_0-aucpr:0.63562
[1900]	validation_0-aucpr:0.63684
[1999]	validation_0-aucpr:0.63761
[0]	validation_0-aucpr:0.36955
[100]	validation_0-aucpr:0.57339
[200]	validation_0-aucpr:0.59178
[300]	validation_0-aucpr:0.60626
[400]	validation_0-aucpr:0.61123
[500]	validation_0-aucpr:0.61617
[600]	validation_0-aucpr:0.61864
[700]	validation_0-aucpr:0.62253
[800]	validation_0-aucpr:0.62484
[90

In [ ]:
params = {'max_depth': 12,
          'learning_rate': 0.025757038240747362,
          'min_child_weight': 10,
          'subsample': 0.9844214806184228,
          'colsample_bytree': 0.6017282633828525,
          'gamma': 0.2863811621422068,
          'reg_alpha': 0.6264287745147197,
          'reg_lambda': 0.11376730360792797}

lean_model = train_xgboost_model(
    X_train_lean, y_train,
    X_test_lean, y_test,
    model_type='classification',
    params=params
)

# 4. Evaluate performance
metrics = evaluate_classification(lean_model, X_test_lean, y_test)
print("\nEvaluation Metrics:", metrics)

[0]	validation_0-aucpr:0.38887
[100]	validation_0-aucpr:0.57083
[200]	validation_0-aucpr:0.59382
[300]	validation_0-aucpr:0.60395
[400]	validation_0-aucpr:0.61026
[500]	validation_0-aucpr:0.61606
[600]	validation_0-aucpr:0.61738
[700]	validation_0-aucpr:0.61989
[800]	validation_0-aucpr:0.62066
[900]	validation_0-aucpr:0.62245
[1000]	validation_0-aucpr:0.62391
[1100]	validation_0-aucpr:0.62426
[1200]	validation_0-aucpr:0.62568
[1300]	validation_0-aucpr:0.62585
[1400]	validation_0-aucpr:0.62716
[1500]	validation_0-aucpr:0.62884
[1600]	validation_0-aucpr:0.62955
[1700]	validation_0-aucpr:0.63077
[1800]	validation_0-aucpr:0.63128
[1900]	validation_0-aucpr:0.63183
[1999]	validation_0-aucpr:0.63201
Threshold 0.20 → Precision: 0.583, Recall: 0.602, F1: 0.592
Threshold 0.25 → Precision: 0.626, Recall: 0.578, F1: 0.601
Threshold 0.30 → Precision: 0.659, Recall: 0.559, F1: 0.605
Threshold 0.35 → Precision: 0.690, Recall: 0.543, F1: 0.608
Threshold 0.40 → Precision: 0.717, Recall: 0.530, F1: 0.60

[0]	validation_0-aucpr:0.37754
[100]	validation_0-aucpr:0.56685
[200]	validation_0-aucpr:0.59356
[300]	validation_0-aucpr:0.60260
[400]	validation_0-aucpr:0.60817
[500]	validation_0-aucpr:0.61340
[600]	validation_0-aucpr:0.61640
[700]	validation_0-aucpr:0.61938
[800]	validation_0-aucpr:0.62125
[900]	validation_0-aucpr:0.62413
[1000]	validation_0-aucpr:0.62688
[1100]	validation_0-aucpr:0.62776
[1200]	validation_0-aucpr:0.62940
[1300]	validation_0-aucpr:0.63072
[1400]	validation_0-aucpr:0.63119
[1500]	validation_0-aucpr:0.63325
[1600]	validation_0-aucpr:0.63431
[1700]	validation_0-aucpr:0.63598
[1800]	validation_0-aucpr:0.63626
[1900]	validation_0-aucpr:0.63722
[1999]	validation_0-aucpr:0.63797
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [17:54:19] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Threshold 0.20 → Precision: 0.622, Recall: 0.595, F1: 0.608
Threshold 0.25 → Precision: 0.665, Recall: 0.570, F1: 0.613
Threshold 0.30 → Precision: 0.694, Recall: 0.552, F1: 0.615
Threshold 0.35 → Precision: 0.722, Recall: 0.536, F1: 0.615
Threshold 0.40 → Precision: 0.744, Recall: 0.522, F1: 0.614
Threshold 0.45 → Precision: 0.765, Recall: 0.510, F1: 0.612
Threshold 0.50 → Precision: 0.782, Recall: 0.498, F1: 0.608
Threshold 0.55 → Precision: 0.800, Recall: 0.487, F1: 0.606
--- Classification Report ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99    140334
           1       0.78      0.50      0.61      5033

    accuracy                           0.98    145367
   macro avg       0.88      0.75      0.80    145367
weighted avg       0.98      0.98      0.98    145367


Evaluation Metrics: {'Accuracy': 0.9778078931256751, 'ROC_AUC': np.float64(0.9336617638364397)}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
import xgboost as xgb

# 1. Combine data back together to allow systematic folding
if isinstance(X_train_lean, pd.DataFrame):
    X_all = pd.concat([X_train_lean, X_test_lean], axis=0).reset_index(drop=True)
    y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)
else:
    X_all = np.vstack((X_train_lean, X_test_lean))
    y_all = np.concatenate((y_train, y_test))

# 2. Initialize an empty array to store the model's validation probabilities
# This will hold the "unbiased test predictions" for the whole dataset
oof_probabilities = np.zeros(len(X_all))

# Configure Stratified 5-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_iterations = []

print("--- Starting Stratified K-Fold Evaluation ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_all, y_all)):
    # Slice into this fold's training and evaluation sets
    X_tr, X_va = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_tr, y_va = y_all.iloc[train_idx], y_all.iloc[val_idx]

    # Initialize your tuned parameters
    fold_model = xgb.XGBClassifier(
        max_depth=12,
        learning_rate=0.0257,
        n_estimators=2000,
        min_child_weight=10,
        subsample=0.9844,
        colsample_bytree=0.6017,
        gamma=0.2863,
        reg_alpha=0.6264,
        reg_lambda=0.1137,
        eval_metric='aucpr',
        device='cuda',
        early_stopping_rounds=150
    )

    # Train the fold
    fold_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

    # Track the best tree iteration for this fold
    best_iterations.append(fold_model.best_iteration)

    # TEST THE MODEL: Predict on the validation fold (data the model hasn't seen yet)
    # and save those probabilities into our global array
    oof_probabilities[val_idx] = fold_model.predict_proba(X_va)[:, 1]

    print(f"Fold {fold + 1} complete. Best iteration: {fold_model.best_iteration}")

print("\n--- Evaluation Phase ---")

In [ ]:
# Test different decision thresholds to evaluate precision-recall behavior
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

for t in thresholds:
    preds = (oof_probabilities >= t).astype(int)

    # Calculate metrics manually for evaluation
    tp = np.sum((preds == 1) & (y_all == 1))
    fp = np.sum((preds == 1) & (y_all == 0))
    fn = np.sum((preds == 0) & (y_all == 1))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Threshold {t:.2f} → Precision: {precision:.3f}, Recall: {recall:.3f}, F1: {f1:.3f}")

Threshold 0.20 → Precision: 0.822, Recall: 0.759, F1: 0.790
Threshold 0.25 → Precision: 0.866, Recall: 0.736, F1: 0.795
Threshold 0.30 → Precision: 0.897, Recall: 0.717, F1: 0.797
Threshold 0.35 → Precision: 0.918, Recall: 0.699, F1: 0.793
Threshold 0.40 → Precision: 0.932, Recall: 0.680, F1: 0.786
Threshold 0.45 → Precision: 0.944, Recall: 0.661, F1: 0.777
Threshold 0.50 → Precision: 0.953, Recall: 0.643, F1: 0.768
Threshold 0.55 → Precision: 0.961, Recall: 0.625, F1: 0.757


In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 9.9 MB/s eta 0:00:00


In [ ]:
import optuna
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import xgboost as xgb

optuna.logging.set_verbosity(optuna.logging.WARNING)

X_all = pd.concat([X_train_lean, X_test_lean], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)

def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 6, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma':            trial.suggest_float('gamma', 0.0, 0.5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.0, 2.0),
        'n_estimators':     2000,
        'eval_metric':      'aucpr',
        'early_stopping_rounds': 100,
        'tree_method':      'hist',
        'device':           'cuda',
        'random_state':     42,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_probs = np.zeros(len(X_all))

    for train_idx, val_idx in skf.split(X_all, y_all):
        X_tr, X_va = X_all.iloc[train_idx], X_all.iloc[val_idx]
        y_tr, y_va = y_all.iloc[train_idx], y_all.iloc[val_idx]

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
        oof_probs[val_idx] = model.predict_proba(X_va)[:, 1]

    # Evaluate at threshold 0.30 (your best threshold)
    preds = (oof_probs >= 0.30).astype(int)
    return f1_score(y_all, preds)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest F1: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

  0%|          | 0/30 [00:00<?, ?it/s]